In [1]:
import pandas as pd
import numpy as np

# Assignment 1: DataFrame Basics

Hi there!

Can you read in the transactions dataset and report on:

* The number of rows and columns
* The names of the columns
* The datatypes of each column

In [2]:
# A common practice is to create a path variable to pass to read_csv
path = r"C:\BIT_Data_analyst\03_python\lectures\10_lesson\transactions.csv"
df = pd.read_csv(path)


In [3]:
df.shape

(83488, 3)

In [4]:
df.columns

Index(['date', 'store_nbr', 'transactions'], dtype='str')

In [5]:
df.dtypes

date              str
store_nbr       int64
transactions    int64
dtype: object

In [6]:
# Data konvertavimas į datetime
# Dažna klaida: palikti datas kaip 'object' ir vėliau gauti keistas filtravimo / grupavimo klaidas.
 
df["date"] = pd.to_datetime(df["date"], errors="coerce")
 
df.dtypes

date            datetime64[us]
store_nbr                int64
transactions             int64
dtype: object

# Assignment 2: Exploring DataFrames

Hello!

* Can you quickly inspect the first 5 rows of the transactions data? 

* Then, dive a bit more deeply into the data and check if there are any missing values.
* What about the number of unique dates? I want to make sure we didn’t leave any out.
* Finally, can you report the mean, median, min and max of “transactions”?  I want to check for any anomalies in our data.


In [7]:
df.head()


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [8]:
df.isna().sum()


date            0
store_nbr       0
transactions    0
dtype: int64

In [9]:
df['date'].nunique()


1682

In [10]:
df['transactions'].agg(['mean', 'median', 'min', 'max'])


mean      1694.602158
median    1393.000000
min          5.000000
max       8359.000000
Name: transactions, dtype: float64

In [11]:
df['transactions'].describe()


count    83488.000000
mean      1694.602158
std        963.286644
min          5.000000
25%       1046.000000
50%       1393.000000
75%       2079.000000
max       8359.000000
Name: transactions, dtype: float64

# Exercise 3 - Accessing DataFrames

Hi, starting to dive deeper into this data.

I noticed that the first row is the only one from 2013-01-01.

* Can you get me a copy of the DataFrame that excludes that row, and only includes “store_nbr” and “transactions”?
* Also, can you report the number of unique store numbers?
* Finally, return the total number of transactions in millions


In [12]:
df2 = df[df['date'] != '2013-01-01'][['store_nbr', 'transactions']]
df2

,store_nbr,transactions
1,1,2111
2,2,2358
3,3,3487
4,4,1922
5,5,1903
...,...,...
83483,50,2804
83484,51,1573
83485,52,2255
83486,53,932


In [13]:
df2['store_nbr'].nunique()

54

In [14]:
df2['transactions'].sum()/1000000

np.float64(141.478175)

# Assignment 4: Dropping Data and Duplicates

Hi there!

Can you:

1. Drop the first row of data? We want it permanently removed. 
2. Drop the date column but not in place
3. Return a dataframe that only includes the last row for each of the stores.

Thanks!

In [15]:
df = df.drop(index=0)

In [16]:
transactions_no_date = df.drop(columns=['date'])

In [17]:
transactions_no_date.columns

Index(['store_nbr', 'transactions'], dtype='str')

In [18]:
last_rows = df.groupby('store_nbr').tail(1)

In [19]:
last_rows

,date,store_nbr,transactions
83434,2017-08-15,1,1693
83435,2017-08-15,2,1737
83436,2017-08-15,3,2956
83437,2017-08-15,4,1283
83438,2017-08-15,5,1310
83439,2017-08-15,6,1589
83440,2017-08-15,7,1780
83441,2017-08-15,8,2621
83442,2017-08-15,9,2155
83443,2017-08-15,10,1010


# Assignment 5: Missing Data

Hello, 

Can you tell if any dates or prices are missing in the oil dataset?

Then compare the mean of the oil series when filling in with mean vs. filling in with 0.

Thanks!

In [20]:
oil = pd.read_csv(r"C:\BIT_Data_analyst\03_python\lectures\10_lesson\oil.csv")

In [21]:
oil.isna().sum()

date           0
dcoilwtico    43
dtype: int64

In [22]:
mean_fill_mean = oil['dcoilwtico'].fillna(oil['dcoilwtico'].mean()).mean()
mean_fill_mean

np.float64(67.71436595744682)

In [23]:
mean_fill_zero = oil['dcoilwtico'].fillna(0).mean()
mean_fill_zero

np.float64(65.32379310344828)

# Assignment 6: Filtering DataFrames

I need some quick research on store 25:

* First, calculate the percentage of times ALL stores had more than 2000 transactions
* Then, calculate the percentage of times store 25 had more than 2000 transactions, and calculate the sum of transactions on these days
* Finally, sum the transactions for stores 25 and 31, that occurred in May or June, and had less than 2000 transactions


In [27]:
df.head()

,date,store_nbr,transactions
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
5,2013-01-02,5,1903


In [ ]:
df.query("date.str[6] == '1'")

In [32]:
# Grupuoji pagal datą ir tikrini, ar VISOS parduotuvės tą dieną > 2000
all_above_2000 = df.groupby("date")["transactions"].apply(lambda x: (x > 2000).all())

percentage_all = all_above_2000.mean() * 100
percentage_all


np.float64(0.0594883997620464)

In [31]:
store25 = df[df["store_nbr"] == 25]

mask_25 = store25["transactions"] > 2000
percentage_25 = mask_25.mean() * 100
sum_25 = store25.loc[mask_25, "transactions"].sum()

percentage_25, sum_25


(np.float64(3.469640644361834), np.int64(144903))

In [34]:
mask_stores = df["store_nbr"].isin([25, 31])
mask_months = df["date"].dt.month.isin([5, 6])
mask_under_2000 = df["transactions"] < 2000

result_sum = df[mask_stores & mask_months & mask_under_2000]["transactions"].sum()
result_sum



np.int64(644910)

# Assignment 7: Sorting DataFrames

Hi there,
* Can you get me a dataset that includes the 5 days with the highest transactions counts? Any similarities between them?
* Then, can you get me a dataset sorted by date from earliest to most recent, but with the highest transactions first and the lowest transactions last for each day?
* Finally, sort the columns in reverse alphabetical order. 

Thanks!


In [35]:
df.head()

,date,store_nbr,transactions
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
5,2013-01-02,5,1903


In [37]:
daily = df.groupby("date")["transactions"].sum()
top5 = daily.sort_values(ascending=False).head(5)
top5


date
2015-12-24    171169
2016-12-24    167542
2016-12-23    156932
2014-12-24    156546
2013-12-24    155846
Name: transactions, dtype: int64

In [39]:
df_sorted = df.sort_values(["date", "transactions"], ascending=[True, False])
df_sorted

,date,store_nbr,transactions
40,2013-01-02,46,4886
38,2013-01-02,44,4821
39,2013-01-02,45,4208
41,2013-01-02,47,4161
11,2013-01-02,11,3547
...,...,...,...
83455,2017-08-15,22,766
83449,2017-08-15,16,742
83465,2017-08-15,32,615
83468,2017-08-15,35,612


In [40]:
df_sorted_cols = df[df.columns.sort_values(ascending=False)]
df_sorted_cols

,transactions,store_nbr,date
1,2111,1,2013-01-02
2,2358,2,2013-01-02
3,3487,3,2013-01-02
4,1922,4,2013-01-02
5,1903,5,2013-01-02
...,...,...,...
83483,2804,50,2017-08-15
83484,1573,51,2017-08-15
83485,2255,52,2017-08-15
83486,932,53,2017-08-15


# Assignment 8: Modifying Columns

Just some quick work, but can you send me the transaction data with the columns renamed?

* Rename `transactions` to `transaction_count` and `store_nbr` to `store_number`.
* Reorder the columns so date is first, then store number, then transaction count.

Thanks!


In [43]:
df_renamed = df.rename(columns={
    "transactions": "transaction_count",
    "store_nbr": "store_number"
})
df_renamed.head()


,date,store_number,transaction_count
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
5,2013-01-02,5,1903


In [45]:
df_final = df_renamed[["date", "store_number", "transaction_count"]]
df_final.head()

,date,store_number,transaction_count
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
5,2013-01-02,5,1903


# Assignment 9: Column Creation

Just some quick work, but can you send me the transaction data with the columns renamed?

* Create a `pct_to_target` column that divides transactions by 2500.
* Then, create a `met_target` column that returns True if `pct_to_target` is greater than or equal to 1.
* Next, create a `bonus_payable` column that equals 100 if `met_target` is True, and 0 if not. Then sum the bonus payable column.
* Finally, create columns for month and day of week as integers. There is some helper code for these dateparts below.



Thanks!



In [47]:
df["pct_to_target"] = df["transactions"] / 2500
df.head()

,date,store_nbr,transactions,pct_to_target
1,2013-01-02,1,2111,0.8444
2,2013-01-02,2,2358,0.9432
3,2013-01-02,3,3487,1.3948
4,2013-01-02,4,1922,0.7688
5,2013-01-02,5,1903,0.7612


In [49]:
df["met_target"] = df["pct_to_target"] >= 1
df.head()

,date,store_nbr,transactions,pct_to_target,met_target
1,2013-01-02,1,2111,0.8444,False
2,2013-01-02,2,2358,0.9432,False
3,2013-01-02,3,3487,1.3948,True
4,2013-01-02,4,1922,0.7688,False
5,2013-01-02,5,1903,0.7612,False


In [50]:
df["bonus_payable"] = df["met_target"].apply(lambda x: 100 if x else 0)
df["bonus_payable"].sum()
df.head()



,date,store_nbr,transactions,pct_to_target,met_target,bonus_payable
1,2013-01-02,1,2111,0.8444,False,0
2,2013-01-02,2,2358,0.9432,False,0
3,2013-01-02,3,3487,1.3948,True,100
4,2013-01-02,4,1922,0.7688,False,0
5,2013-01-02,5,1903,0.7612,False,0


In [52]:
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

df.head()

,date,store_nbr,transactions,pct_to_target,met_target,bonus_payable,month,day_of_week
1,2013-01-02,1,2111,0.8444,False,0,1,2
2,2013-01-02,2,2358,0.9432,False,0,1,2
3,2013-01-02,3,3487,1.3948,True,100,1,2
4,2013-01-02,4,1922,0.7688,False,0,1,2
5,2013-01-02,5,1903,0.7612,False,0,1,2


# Assignment 10: np.select

Hi there! I need a few columns created.

1. Create a ‘seasonal_bonus’ column that applies to these dates: 
    * All days in December (month = 12)
    * Sundays (day_of_week = 6) in May (month = 5)
    * Mondays (day_of_week = 0) in July (month = 7)
2. Call the December bonus ‘Holiday Bonus’, the May bonus ‘Corporate Month’, and the July bonus ‘Summer Special’. If no bonus applies, the column should display ‘None’. 
3. Finally, calculate the total bonus owed at $100 per day.

Thanks!

In [57]:
cond_dec = (df["month"] == 12)
cond_may = (df["month"] == 5) & (df["day_of_week"] == 6)
cond_jul = (df["month"] == 7) & (df["day_of_week"] == 0)


In [58]:
conditions = [cond_dec, cond_may, cond_jul]
choices = ["Holiday Bonus", "Corporate Month", "Summer Special"]



In [64]:
df["seasonal_bonus"] = np.select(conditions, choices, default="None")
df["bonus_amount"] = np.where(df["seasonal_bonus"] == "None", 0, 100)
df["bonus_amount"].sum()
df.head()



,date,store_nbr,transactions,pct_to_target,met_target,bonus_payable,month,day_of_week,seasonal_bonus,bonus_amount
1,2013-01-02,1,2111,0.8444,False,0,1,2,None,0
2,2013-01-02,2,2358,0.9432,False,0,1,2,None,0
3,2013-01-02,3,3487,1.3948,True,100,1,2,None,0
4,2013-01-02,4,1922,0.7688,False,0,1,2,None,0
5,2013-01-02,5,1903,0.7612,False,0,1,2,None,0


# Assignment 11: Assign 

* Drop the columns that have been created so far (keep only date, store_number, and transaction count), and recreate them using the assign method.
* Then sum the seasonal bonus owed once again to make sure the numbers are correct.


In [66]:
# Drop columns we created in prior exercises
df = df[["date", "store_nbr", "transactions"]]
df.head()

,date,store_nbr,transactions
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
5,2013-01-02,5,1903


In [68]:
# Create same columns with assign
df = df.assign(month = df["date"].dt.month)
df.head()

,date,store_nbr,transactions,month
1,2013-01-02,1,2111,1
2,2013-01-02,2,2358,1
3,2013-01-02,3,3487,1
4,2013-01-02,4,1922,1
5,2013-01-02,5,1903,1


In [ ]:
df = df.assign(day_of_week = df["date"].dt.dayofweek)
df = df.assign(pct_to_target = df["transactions"] / 2500)
df = df.assign(met_target = df["pct_to_target"] >= 1)
df = df.assign(seasonal_bonus = np.select(conditions, choices, default="None"))
df = df.assign(bonus_amount = np.where(df["seasonal_bonus"] == "None", 0, 100))


df.head()

,date,store_nbr,transactions,month,day_of_week,pct_to_target,met_target,seasonal_bonus
1,2013-01-02,1,2111,1,2,0.8444,False,None
2,2013-01-02,2,2358,1,2,0.9432,False,None
3,2013-01-02,3,3487,1,2,1.3948,True,None
4,2013-01-02,4,1922,1,2,0.7688,False,None
5,2013-01-02,5,1903,1,2,0.7612,False,None


# Assignment 12: Memory Optimization

Reduce the memory usage of the transactions DataFrame to below 5MB.

In [72]:
df.dtypes

date              datetime64[us]
store_nbr                  int64
transactions               int64
month                      int32
day_of_week                int32
pct_to_target            float64
met_target                  bool
seasonal_bonus               str
dtype: object

In [77]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 83487 entries, 1 to 83487
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            83487 non-null  datetime64[us]
 1   store_nbr       83487 non-null  int16         
 2   transactions    83487 non-null  int16         
 3   month           83487 non-null  int8          
 4   day_of_week     83487 non-null  int8          
 5   pct_to_target   83487 non-null  float64       
 6   met_target      83487 non-null  bool          
 7   seasonal_bonus  83487 non-null  category      
dtypes: bool(1), category(1), datetime64[us](1), float64(1), int16(2), int8(2)
memory usage: 1.9 MB


In [76]:
df["store_nbr"] = df["store_nbr"].astype("int16")
df["transactions"] = df["transactions"].astype("int16")
df["month"] = df["month"].astype("int8")
df["day_of_week"] = df["day_of_week"].astype("int8")
df["met_target"] = df["met_target"].astype("bool")
df["seasonal_bonus"] = df["seasonal_bonus"].astype("category")

